# Crowd Counting with CSRNet and OpenVINO™

[CSRNet](https://arxiv.org/abs/1802.10062) (*Dilated Convolutional Neural Networks
for Understanding the Highly Congested Scenes*, CVPR 2018) estimates the number of
people in an image by regressing a **density map** whose integral equals the crowd
count. It pairs a VGG-16 front-end with a dilated-convolution back-end that keeps a
large receptive field without losing spatial resolution.

This notebook shows how to:

1. Set up a dedicated virtual environment **`.csrnet-nb-venv`** with every dependency.
2. Load a pretrained CSRNet model (PyTorch).
3. Convert it to **OpenVINO IR** at **FP32** and **FP16**.
4. Quantize it to **INT8** with **NNCF** post-training quantization.
5. Run inference on **CPU / GPU / NPU** and visualize the predicted density map.
6. Evaluate the result with **count error, PSNR and SSIM** against the ground truth.

For illustration, we use a single test image from **ShanghaiTech Part A** whose **ground-truth count is 141**.

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).
#### Table of contents:

- [Installation Instructions](#Installation-Instructions)
- [Install dependencies](#Install-dependencies)
- [Imports](#Imports)
- [Download the model](#Download-the-model)
- [Preprocessing and ground-truth density map](#Preprocessing-and-ground-truth-density-map)
- [Convert to OpenVINO IR (FP32 and FP16)](#Convert-to-OpenVINO-IR-(FP32-and-FP16))
- [Quantize to INT8 with NNCF](#Quantize-to-INT8-with-NNCF)
- [Select inference device](#Select-inference-device)
- [Run inference](#Run-inference)
- [Visualize density maps](#Visualize-density-maps)
- [Quantitative evaluation](#Quantitative-evaluation)
- [Latency benchmark](#Latency-benchmark)
- [Conclusion](#Conclusion)
<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/csrnet/csrnet.ipynb" />

## Installation Instructions
[back to top ⬆️](#Table-of-contents)
 
This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start. For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Install dependencies
[back to top ⬆️](#Table-of-contents)

This notebook is **self-contained**: the cell below pip-installs everything it
needs directly into the running Jupyter kernel.
It follows the OpenVINO-notebook methodology (a small `pip_install` helper that
shells out to the active interpreter).

- **torch / torchvision** are installed from the **Intel XPU index**
  (`--extra-index-url https://download.pytorch.org/whl/xpu`) so the `+xpu` build
  is selected. If a non-XPU torch is already present, the cell force-installs the
  matching `+xpu` build.
- Re-runs are fast: pip skips packages that are already satisfied.

> **Virtual environment.** The notebook is meant to run in a dedicated virtual
> environment, **`.csrnet-nb-venv`**, registered as a Jupyter kernel. If it is not
> set up yet, create it once from a terminal in this folder:
>
> ```bash
> python3 -m venv .csrnet-nb-venv
> .csrnet-nb-venv/bin/python -m pip install -U pip ipykernel
> .csrnet-nb-venv/bin/python -m ipykernel install --user --name csrnet-nb-venv --display-name "Python (.csrnet-nb-venv)"
> ```
>
> Then select the **Python (.csrnet-nb-venv)** kernel in the notebook
> (*Kernel ▸ Change Kernel*) and re-run from the top. If you already have a
> suitable environment, you can instead just run the cell below in the active
> kernel.

In [ ]:
# === Install dependencies into the active Jupyter kernel =====================
# This notebook is self-contained: it pip-installs everything it needs into the
# running kernel. Re-runs are fast because pip skips
# packages that are already satisfied.
import subprocess
import sys


def pip_install(*args):
    """Install packages into the active kernel (OpenVINO-notebook methodology)."""
    cli = []
    for a in args:
        cli.extend(str(a).split(" "))
    subprocess.run([sys.executable, "-m", "pip", "install", *cli], check=True)


# --- torch / torchvision: Intel XPU build via the XPU index ------------------
# The XPU index carries `torch<ver>+xpu`; its local-version tag sorts above the
# plain PyPI wheel, so `pip install torch` with this extra index resolves to the
# XPU build. If a non-XPU torch is already present, force the XPU build in.
try:
    import torch as _t

    _torch_ver = _t.__version__
except Exception:
    _torch_ver = None

if _torch_ver is None:
    pip_install("-q", "--extra-index-url", "https://download.pytorch.org/whl/xpu", "torch>=2.1", "torchvision")
elif "+xpu" not in _torch_ver:
    _base = _torch_ver.split("+")[0]
    print(f"torch {_torch_ver} is not an XPU build -> installing torch=={_base}+xpu")
    pip_install("-q", "--force-reinstall", "--no-deps", "--extra-index-url", "https://download.pytorch.org/whl/xpu", f"torch=={_base}+xpu", "torchvision")
    pip_install("-q", "--extra-index-url", "https://download.pytorch.org/whl/xpu", f"torch=={_base}+xpu")  # pull the XPU runtime deps
else:
    print(f"torch {_torch_ver} already present (XPU build)")

# --- the rest of the runtime --------------------------------------------------
pip_install(
    "-q",
    "openvino>=2024.4.0",
    "nncf>=2.13.0",
    "opencv-python",
    "scipy",
    "scikit-image>=0.21",
    "matplotlib",
    "Pillow",
    "tqdm",
    "ipywidgets>=8",
    "requests",
    "gdown>=5.1.0",
)

import importlib

print("\ndependencies ready in", sys.executable)
for m in ["openvino", "nncf", "torch", "torchvision", "cv2", "scipy", "skimage", "PIL", "numpy", "matplotlib", "tqdm", "requests", "gdown"]:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:<16} {getattr(mod, '__version__', 'ok')}")
    except Exception as exc:
        print(f"  {m:<16} MISSING -> {exc}")

# --- telemetry (openvino_notebooks CI) ----------------------------------
# Fetch the shared helper and report that this notebook was run.
import urllib.request
from pathlib import Path

if not Path("notebook_utils.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
        "notebook_utils.py",
    )

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("csrnet.ipynb")

## Imports
[back to top ⬆️](#Table-of-contents)

In [ ]:
import warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import scipy.spatial
from scipy.io import loadmat
from scipy.ndimage import gaussian_filter
from PIL import Image

import torch
import torch.nn as nn

import openvino as ov
import nncf
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

warnings.filterwarnings("ignore")
nncf.set_log_level(40)  # quieter NNCF logs
%matplotlib inline

core = ov.Core()
DATA_DIR = Path("data")
MODEL_DIR = Path("model")
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
print("OpenVINO:", ov.__version__)

## Download the model
[back to top ⬆️](#Table-of-contents)

This example needs **three files**, all fetched into `data/` on first run and reused afterwards:

| File | What it is | Source |
|---|---|---|
| `IMG_114.jpg` | ShanghaiTech Part A test image (GT count = 141) | [release asset](https://github.com/nagabh/openvino_notebooks/releases/download/csrnet-data-v1/IMG_114.jpg) |
| `GT_IMG_114.mat` | head-point annotations for the ground-truth density map | [release asset](https://github.com/nagabh/openvino_notebooks/releases/download/csrnet-data-v1/GT_IMG_114.mat) |
| `PartAmodel_best.pth.tar` | pretrained CSRNet Part A weights | [Google Drive](https://drive.google.com/file/d/1Z-atzS5Y2pOd-nEWqZRVBDMYJDreGWHH/view) — the CSRNet authors' [reference repo](https://github.com/leeyeehoo/CSRNet-pytorch) |

The test image and its ground-truth annotations come from the **ShanghaiTech Part A** dataset, which is gated, so the two small files are hosted as a [release asset](https://github.com/nagabh/openvino_notebooks/releases/tag/csrnet-data-v1) and downloaded with `requests`. The weights are fetched from the authors' Google Drive with `gdown`. Every download is verified against a known **SHA-256** and is skipped when a valid copy already exists.

The full ShanghaiTech dataset can be found [here.](https://drive.google.com/file/d/16dhJn7k4FWVwByRsQAEpl9lwjuV03jVI/view)


In [ ]:
import hashlib

import gdown
import requests

# The test image + ground-truth annotations come from the (gated) ShanghaiTech Part A
# dataset, so the two small files are hosted as a release asset and fetched with requests.
# The pretrained weights are pulled from the authors' Google Drive with gdown.
# Direct link: https://drive.google.com/file/d/1Z-atzS5Y2pOd-nEWqZRVBDMYJDreGWHH/view
WEIGHTS_GDRIVE_ID = "1Z-atzS5Y2pOd-nEWqZRVBDMYJDreGWHH"
DATA_RELEASE = "https://github.com/nagabh/openvino_notebooks/releases/download/csrnet-data-v1"

# filename -> expected SHA-256 (pins the exact file and guards against corruption)
SHA256 = {
    "IMG_114.jpg": "8fc39904994bd8b1c1ae1e58bbb6232bf108d362e80b22fc203fe3021f020418",
    "GT_IMG_114.mat": "216897f8585fa141bf075d399accd6bbbe9cb8e51bab9336493e87c83ac6ae31",
    "PartAmodel_best.pth.tar": "68383c1053be371ad54b0061ab99fed08c2bfff39de73937f2a4388041ab0128",
}
GT_COUNT = 141  # ground-truth count for this scene


def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _verify(path, name):
    expected = SHA256.get(name)
    if expected and _sha256(path) != expected:
        Path(path).unlink(missing_ok=True)
        raise ValueError(f"checksum mismatch for {name} - download corrupted or wrong source")
    return path


def fetch_url(name, url):
    """Download a file from a plain HTTP(S) URL into data/ (skipped if a valid copy exists)."""
    path = DATA_DIR / name
    if path.exists() and _sha256(path) == SHA256[name]:
        print(f"✓ found {path}")
        return path
    print(f"↓ downloading {name} from {url}")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    path.write_bytes(resp.content)
    _verify(path, name)
    print(f"✓ saved {path}")
    return path


def fetch_weights():
    """Download the CSRNet Part A weights from Google Drive."""
    name = "PartAmodel_best.pth.tar"
    path = DATA_DIR / name
    if path.exists() and _sha256(path) == SHA256[name]:
        print(f"✓ found {path}")
        return path
    print(f"↓ downloading {name} from Google Drive (id={WEIGHTS_GDRIVE_ID})")
    gdown.download(id=WEIGHTS_GDRIVE_ID, output=str(path), quiet=False)
    _verify(path, name)
    print(f"✓ saved {path}")
    return path


IMAGE_FILE = fetch_url("IMG_114.jpg", f"{DATA_RELEASE}/IMG_114.jpg")
ANNOT_FILE = fetch_url("GT_IMG_114.mat", f"{DATA_RELEASE}/GT_IMG_114.mat")
WEIGHTS_FILE = fetch_weights()

In [ ]:
def make_layers(cfg, in_channels=3, dilation=False):
    rate = 2 if dilation else 1
    layers = []
    for v in cfg:
        if v == "M":
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            layers += [nn.Conv2d(in_channels, v, kernel_size=3, padding=rate, dilation=rate), nn.ReLU(inplace=True)]
            in_channels = v
    return nn.Sequential(*layers)


class CSRNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.frontend = make_layers([64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512])
        self.backend = make_layers([512, 512, 512, 256, 128, 64], in_channels=512, dilation=True)
        self.output_layer = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x


def load_pretrained(weights_path):
    model = CSRNet()
    ckpt = torch.load(str(weights_path), map_location="cpu", weights_only=False)
    state = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    clean = {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}
    model.load_state_dict(clean, strict=True)
    model.eval()
    return model


torch_model = load_pretrained(WEIGHTS_FILE)
print("CSRNet loaded — parameters:", sum(p.numel() for p in torch_model.parameters()))

## Preprocessing and ground-truth density map
[back to top ⬆️](#Table-of-contents)

The model expects an RGB image normalized with ImageNet statistics. Spatial dims are
cropped to a multiple of 8 (three 2× pooling stages).

The **ground-truth density map** for ShanghaiTech Part A is built from the head-point
annotations using a *geometry-adaptive Gaussian kernel* (σ scaled by the mean distance
to each head's 3 nearest neighbours), exactly as in the CSRNet paper. Its sum equals
the number of annotated heads.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)


def preprocess(pil_img, factor=8):
    x = np.asarray(pil_img.convert("RGB"), dtype=np.float32) / 255.0  # HWC
    x = np.transpose(x, (2, 0, 1))[None]  # 1CHW
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    _, _, h, w = x.shape
    x = x[:, :, : h - h % factor, : w - w % factor]  # crop to /8
    return np.ascontiguousarray(x, dtype=np.float32)


def gt_density_map(shape, points):
    # Geometry-adaptive (k-NN) Gaussian density map - ShanghaiTech Part A.
    h, w = shape
    density = np.zeros((h, w), dtype=np.float32)
    pts = np.asarray(points, dtype=np.float32)
    n = len(pts)
    if n == 0:
        return density
    tree = scipy.spatial.KDTree(pts.copy(), leafsize=2048)
    distances, _ = tree.query(pts, k=min(4, n))
    for i, p in enumerate(pts):
        x = min(w - 1, max(0, int(round(p[0]))))
        y = min(h - 1, max(0, int(round(p[1]))))
        pt = np.zeros((h, w), dtype=np.float32)
        pt[y, x] = 1.0
        if n > 3:
            sigma = (distances[i][1] + distances[i][2] + distances[i][3]) * 0.1
        else:
            sigma = np.average([h, w]) / 4.0
        density += gaussian_filter(pt, sigma, mode="constant")
    return density


# Load image + annotations
pil_image = Image.open(IMAGE_FILE)
image_rgb = np.asarray(pil_image.convert("RGB"))
points = loadmat(ANNOT_FILE)["image_info"][0][0][0][0][0]

input_tensor = preprocess(pil_image)
gt_density = gt_density_map(image_rgb.shape[:2], points)

print(f"image size (HxW): {image_rgb.shape[:2]}")
print(f"model input shape: {tuple(input_tensor.shape)}")
print(f"annotated heads (ground-truth count): {len(points)}")
print(f"ground-truth density map sum: {gt_density.sum():.2f}")

## Convert to OpenVINO IR (FP32 and FP16)
[back to top ⬆️](#Table-of-contents)

`ov.convert_model` traces the PyTorch model. We keep the spatial dimensions **dynamic**
(`[1, 3, ?, ?]`) so any image resolution works. Saving with `compress_to_fp16=True`
stores the weights in half precision (≈2× smaller) without changing the graph.

In [ ]:
IR_FP32 = MODEL_DIR / "csrnet_fp32.xml"
IR_FP16 = MODEL_DIR / "csrnet_fp16.xml"

example = torch.zeros(1, 3, 768, 1024, dtype=torch.float32)
ov_model = ov.convert_model(torch_model, example_input=example, input=[[1, 3, -1, -1]])
ov_model.inputs[0].get_tensor().set_names({"image"})
ov_model.outputs[0].get_tensor().set_names({"density_map"})

ov.save_model(ov_model, IR_FP32, compress_to_fp16=False)
ov.save_model(ov_model, IR_FP16, compress_to_fp16=True)
print("FP32 IR:", round(IR_FP32.with_suffix(".bin").stat().st_size / 1e6, 1), "MB")
print("FP16 IR:", round(IR_FP16.with_suffix(".bin").stat().st_size / 1e6, 1), "MB")

## Quantize to INT8 with NNCF
[back to top ⬆️](#Table-of-contents)

NNCF post-training quantization needs a small **calibration dataset** to record
activation ranges. Production use should calibrate on tens to hundreds of
representative images; since this notebook downloads only a single image, we build a
lightweight calibration set from random crops of that image purely to demonstrate the
INT8 flow.

In [ ]:
import inspect

# Set QUANTIZE = True to (re)run NNCF post-training quantization.
# Keep it False for normal "Run All" sessions to reuse the existing INT8 IR.
QUANTIZE = False
# Keep the final 1x1 density-map conv in FP32 (selective quantization).
# Default in ov_quantize.py: enabled for Part B only; off here (Part A).
KEEP_OUTPUT_FP = False
NUM_CALIB = 50  # Part A default (ov_quantize.py: A=50, B=200)

IR_INT8 = MODEL_DIR / "csrnet_int8.xml"

if QUANTIZE or not IR_INT8.exists():
    # ShanghaiTech Part A training images (same dataset root as ov_scripts/config.py).
    DATASET_ROOT = Path("../../datasets/ShanghaiTech_Crowd_Counting_Dataset")
    TRAIN_IMG_DIR = DATASET_ROOT / "part_A_final" / "train_data" / "images"
    if not TRAIN_IMG_DIR.exists():
        raise FileNotFoundError(f"ShanghaiTech dataset not found at {DATASET_ROOT.resolve()}. " "Create a link at ./datasets pointing to the dataset root.")

    def make_calibration_set(img_dir, n=NUM_CALIB):
        """Evenly spaced subset of training images, preprocessed to NCHW float32."""
        files = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
        idx = np.linspace(0, len(files) - 1, min(n, len(files))).astype(int)
        return [preprocess(Image.open(files[i])) for i in idx]

    def make_nncf_dataset(inputs):
        """Build an nncf.Dataset across NNCF versions (transform_fn -> transform_func in 3.3+)."""

        def transform_fn(x):
            return x  # feed each preprocessed image directly to the model input

        params = inspect.signature(nncf.Dataset).parameters
        if "transform_func" in params:
            return nncf.Dataset(inputs, transform_func=transform_fn)
        return nncf.Dataset(inputs, transform_fn=transform_fn)

    print(f"[quantize] building calibration set ({NUM_CALIB} training images) ...")
    calib = make_calibration_set(TRAIN_IMG_DIR)
    print(f"[quantize] calibration samples: {len(calib)}")
    calibration_dataset = make_nncf_dataset(calib)

    ignored_scope = None
    if KEEP_OUTPUT_FP:
        # Keep the final 1x1 conv (density map) in FP32; everything else INT8.
        ignored_scope = nncf.IgnoredScope(patterns=[r"__module\.output_layer"])
        print("[quantize] keeping output conv in FP32 (selective quantization)")

    print("[quantize] running NNCF post-training quantization (INT8) ...")
    int8_model = nncf.quantize(
        core.read_model(IR_FP32),
        calibration_dataset,
        preset=nncf.QuantizationPreset.MIXED,
        target_device=nncf.TargetDevice.GPU,
        subset_size=len(calib),
        ignored_scope=ignored_scope,
    )
    ov.save_model(int8_model, IR_INT8)
    print("INT8 IR:", round(IR_INT8.with_suffix(".bin").stat().st_size / 1e6, 1), "MB")
else:
    print(f"✓ found {IR_INT8} (set QUANTIZE = True to re-quantize)")

## Select inference device
[back to top ⬆️](#Table-of-contents)

Pick the OpenVINO device. `AUTO` lets the runtime choose the best available target
(GPU/NPU/CPU).

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO")
device

## Run inference
[back to top ⬆️](#Table-of-contents)

First run the original **PyTorch** model to obtain a reference density map, then
compile each OpenVINO precision on the selected device. In every case, the count is simply the sum of the density map. The PyTorch result is considered as the baseline and the FP32/FP16/INT8 OpenVINO IRs are compared against that.

In [ ]:
# Reference inference with the original PyTorch model.
def infer_torch(model, tensor):
    with torch.no_grad():
        out = model(torch.from_numpy(tensor))
    return out[0, 0].cpu().numpy()  # HxW density map


# Reference inference with the OpenVINO models.
def infer(ir_path, tensor, device_name, precision="FP32"):
    config = {}
    hint = _PRECISION_HINTS.get(precision)
    if hint is not None:
        config["INFERENCE_PRECISION_HINT"] = hint
    compiled = core.compile_model(core.read_model(ir_path), device_name, config)
    return compiled(tensor)[compiled.output(0)][0, 0]  # HxW density map


# INFERENCE_PRECISION_HINT tells the device which compute precision to use.
# It only accepts floating-point hints (f32 / f16 / bf16); "int8" is NOT a
# valid value, so for the INT8 model we omit the hint and let the device run
# the already-quantized weights at their native precision.
_PRECISION_HINTS = {"FP32": "f32", "FP16": "f16", "INT8": None}

pred_maps = {"PyTorch": infer_torch(torch_model, input_tensor)}
precisions = {"FP32": IR_FP32, "FP16": IR_FP16, "INT8": IR_INT8}
pred_maps.update({name: infer(path, input_tensor, device.value, name) for name, path in precisions.items()})

for name, dmap in pred_maps.items():
    print(f"{name:<8}: predicted count = {dmap.sum():7.2f}   (density map {dmap.shape})")

# PyTorch baseline first, then the OpenVINO precisions.pred_maps = {"PyTorch": infer_torch(torch_model, input_tensor)}

## Visualize density maps
[back to top ⬆️](#Table-of-contents)

For illustration, we have the input image, the ground-truth density map, and the OpenVINO **FP16** density map. Brighter regions correspond to higher person density.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
axes[0].imshow(image_rgb)
axes[0].set_title("Input image")
axes[1].imshow(gt_density, cmap="jet")
axes[1].set_title(f"Ground truth  (count = {gt_density.sum():.0f})")
axes[2].imshow(pred_maps["FP16"], cmap="jet")
axes[2].set_title(f"CSRNet FP16  (count = {pred_maps['FP16'].sum():.0f})")
for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Quantitative evaluation
[back to top ⬆️](#Table-of-contents)

We report, for every precision:

- **GT count** and **estimated count** (sum of the density map) and the absolute error.
- **PSNR** and **SSIM** between the predicted and ground-truth density maps — as used in the CSRNet paper. The prediction (1/8 resolution) is resized to the ground-truth resolution with count preserved, then both maps are normalized to a shared `[0, 1]` scale before computing the metrics.

In [ ]:
def density_quality(pred_map, gt_map):
    gh, gw = gt_map.shape
    pred_r = cv2.resize(pred_map.astype(np.float32), (gw, gh), interpolation=cv2.INTER_CUBIC)
    if pred_r.sum() > 0:  # preserve the total count after resize
        pred_r *= pred_map.sum() / pred_r.sum()
    scale = max(gt_map.max(), pred_r.max(), 1e-8)
    gt_n = (gt_map / scale).astype(np.float32)
    pred_n = np.clip(pred_r / scale, 0, 1).astype(np.float32)
    psnr = peak_signal_noise_ratio(gt_n, pred_n, data_range=1.0)
    ssim = structural_similarity(gt_n, pred_n, data_range=1.0)
    return psnr, ssim


header = f"{'Precision':<10}{'GT':>6}{'Estimate':>11}{'|Error|':>9}{'PSNR (dB)':>11}{'SSIM':>8}"
print(header)
print("-" * len(header))
for name, dmap in pred_maps.items():
    est = float(dmap.sum())
    psnr, ssim = density_quality(dmap, gt_density)
    print(f"{name:<10}{GT_COUNT:>6}{est:>11.2f}{abs(est - GT_COUNT):>9.2f}{psnr:>11.2f}{ssim:>8.4f}")

## Latency benchmark
[back to top ⬆️](#Table-of-contents)

A quick steady-state latency measurement per precision on the selected
device (fixed input shape).

In [ ]:
import time


def benchmark_ov(ir_path, tensor, device_name, runs=20, warmup=5):
    compiled = core.compile_model(core.read_model(ir_path), device_name, {"PERFORMANCE_HINT": "LATENCY"})
    for _ in range(warmup):
        compiled(tensor)
    t0 = time.perf_counter()
    for _ in range(runs):
        compiled(tensor)
    return (time.perf_counter() - t0) / runs * 1e3


def benchmark_torch(model, tensor, runs=20, warmup=5):
    with torch.no_grad():
        for _ in range(warmup):
            model(torch.from_numpy(tensor))
        t0 = time.perf_counter()
        for _ in range(runs):
            model(torch.from_numpy(tensor))
    return (time.perf_counter() - t0) / runs * 1e3


print(f"{'Precision':<10}{'Latency (ms)':>14}")
print("-" * 24)
print(f"{'PyTorch':<10}{benchmark_torch(torch_model, input_tensor):>14.2f}")
for name, path in precisions.items():
    print(f"{name:<10}{benchmark_ov(path, input_tensor, device.value):>14.2f}")

## Conclusion
[back to top ⬆️](#Table-of-contents)

We just converted CSRNet to OpenVINO IR (FP32/FP16), quantized it to INT8 with NNCF, ran
it on the selected device, and evaluated the prediction on a scene (test image) from ShanghaiTech Part A 
(GT = 141) using count error, PSNR and SSIM. FP32/FP16/INT8 OpenVINO IRs match the PyTorch model's count closely with INT8 offering upto 2x speedup.

### References
- CSRNet paper: https://arxiv.org/abs/1802.10062
- ShanghaiTech dataset (Zhang *et al.*, CVPR 2016)
- OpenVINO: https://docs.openvino.ai
- NNCF: https://github.com/openvinotoolkit/nncf